# Einstein's riddle

<figure>
<center>
<img src='https://i.ytimg.com/vi/1rDVz_Fb6HQ/maxresdefault.jpg'
 />
<figcaption></figcaption></center>
</figure>

<figure>
<center>
<img src='https://i.ibb.co/ydDP5R3/einstein.png'
/>
<figcaption></figcaption></center>
</figure>


**Einstein's riddle**, also known as the Zebra Puzzle, is a well-known logic puzzle. It is sometimes attributed to Albert Einstein (hence the name) and sometimes to Lewis Carroll, although its true authorship remains uncertain.

It involves five houses in a row:

`House 1 | House 2 | House 3 | House 4 | House 5`


Each of the five houses is painted a unique color (that is, no two houses have the same color). Each inhabitant of the house has a unique nationalities, type of pet, beverage they drink, and brand of cigarettes they smoke.

Moreover, the puzzle provides 15 clues or constraints:

1. The Brit lives in the Red house.

2. The Swede keeps Dogs as pets.

3. The Dane drinks Tea.

4. The Green house is exactly to the left of the White house.

5. The owner of the Green house drinks Coffee.

6. The person who smokes Pall Mall rears Birds.

7. The owner of the Yellow house smokes Dunhill.

8. The man living in the centre house drinks Milk.

9. The Norwegian lives in the first house.

10. The man who smokes Blends lives next to the one who keeps Cats.

11. The man who keeps Horses lives next to the man who smokes Dunhill.

12. The man who smokes Blue Master drinks Beer.

13. The German smokes Prince.

14. The Norwegian lives next to the Blue house.

15. The man who smokes Blends has a neighbour who drinks Water.


Now the questions: **Who owns the fish?** And **who drinks water?**. (Other versions have zebras as possible pets, and ask you to determine who owns the zebra; hence the name "Zebra puzzle").

Check out the puzzle also in the link below, which is helpful to try out solutions by hand. The possible values for colors, nationalities, etc., can be found in the dropdown menus of that table.

https://www.brainzilla.com/logic/zebra/einsteins-riddle/




Einstein's riddle has been used as a benchmark in the evaluation of algorithms for solving [*constraint satisfaction problems*](https://en.wikipedia.org/wiki/Constraint_satisfaction#:~:text=In%20artificial%20intelligence%20and%20operations,that%20the%20variables%20must%20satisfy.) (CSPs). CSPs are a fundamental topic in artificial intelligence. With this exercise, you get to work a bit on a CSP problem, using logic.

A CSP is a type of computational problem where you have a set of variables, which can take certain values. The goal is to figure out what values to assign to them while making sure you follow certain constraints about which combinations of values are allowed. Einstein's riddle is a CSP because it involves a set of variables (colors of houses, nationalities, pets, drinks, and cigarette brands) with specific constraints (e.g., each house must have a different nationality, color, pet, drink, and cigarette brand).

One approach to tackle certain types of CSP problems is by using propositional logic. In this case, propositional atoms can be used to represent the possible attributes of the houses, and the clues/constraints can be expressed by logical formulas. Translating the puzzle in propositional allows us to use an inference procedure to solve it.

**Your mission is to write a set of propositional formulas that faithfully describe the constraints in the problem. After that, you'll feed the formulas to a propositional logic solver to derive the solution to the puzzle.**

# 1. Helper functions and syntax

We'll define some helper functions, which can be used to write the logical formulas defining the clues and constraints.

You have the option to use these functions or to craft the formulas manually, depending on your preference. Since certain formulas can be quite lengthy, these functions are designed to simplify the process, but it's entirely acceptable to create the expressions manually if you prefer.

However, it is mandatory to use the following syntax for logical connectives:
- negation symbols: `-`, or `~`
- conjunction symbols: `&`
- disjunction symbols: `|`, `v`, `V`
- conditional symbols: `->`, `=>`
- biconditional symbols: `<->`, `<=>`

This ensures that the solver can understand your formulas.

There is no operator precedence; all operators are bound from left:
`a & b v c & d v e` is read as `((((a & b) v c) & d) v e)`.

To write conjunctions or disjunctions involving several atoms, you may use the following functions.

## Ors and Ands

In [1]:
atoms = ["a","b","c","d"]

def big_or(atoms):
  return f"({' v '.join(atoms) })"
def big_and(atoms):
  return f"({ ' & '.join(atoms) })"

Try the functions: print the "or" and the "and" for `atoms = ["a","b","c","d"]`

In [2]:
# your code here
print("Big OR:", big_or(atoms))
print("Big AND:", big_and(atoms))

Big OR: (a v b v c v d)
Big AND: (a & b & c & d)


# 1. Helper functions and syntax

## Mutual Exclusion:  "Only one  of a list of atoms is true¨
The following function takes a list of atoms as input and returns a formula expressing that one atom is true if and only if the other are false; that is, only one atom in the list can be true.


In [3]:
def mutually_exclusive(atoms):

  # Initialize an empty list to store the subformulas
  subformulas = []

  # Generate the subformulas for mutual exclusivity
  for atom in atoms:
    other_atoms = [a for a in atoms if a != atom]
    exclusivity_formula = f"({atom} <=> ({' & '.join(['-' + a for a in other_atoms])}))"
    subformulas.append(exclusivity_formula)

  # Combine all the equivalences into a single conjunction
  mutual_exclusivity_formula = ' & '.join(subformulas)
  return mutual_exclusivity_formula

Try it: Print the mutual exclusion clause of the same list of atoms as before.

In [4]:
# your code here
print("Mutual exclusion clause:")
print(mutually_exclusive(atoms))

Mutual exclusion clause:
(a <=> (-b & -c & -d)) & (b <=> (-a & -c & -d)) & (c <=> (-a & -b & -d)) & (d <=> (-a & -b & -c))


## Exactly one is true
The following function takes a list of atoms as input, and retuns a formula stating that *excactly one* of the atoms in the list is true.

In [6]:
def exactly_one_condition(atoms):
    n = len(atoms)
    # Create the condition that at least one of the atoms is true
    at_least_one_true = f"{ ' v '.join(atoms) }"
    # Create the condition that at most one is true
    false_conditions = []
    for i in range(n):
      other_atoms = atoms[:i] + atoms[i+1:]
      false_conditions.append(f"({' & '.join(['-' + a for a in other_atoms])})")
    at_most_one_true = f"{' v '.join(false_conditions)}"
    # Combine the conditions into a conjunction
    exactly_one_condition_formula = f"({at_least_one_true}) & ({at_most_one_true})"
    return exactly_one_condition_formula

Try it: Make the clause for the "Exactly one Condition" for `atoms = ["a","b","c","d"]`

In [7]:
# your code here
print("Exactly one condition clause:")
print(exactly_one_condition(atoms))

Exactly one condition clause:
(a v b v c v d) & ((-b & -c & -d) v (-a & -c & -d) v (-a & -b & -d) v (-a & -b & -c))


## Join and print
The following function simply takes a list of formulas and prints the conjunction of the formulas.

In [8]:
def conjoin_and_print(formulas):
  print(' & \n'.join(formulas))

Try it. Generate again the mutually exclusive and the exactly one conditions, conjoin them into a larger formula, and print out the result.

In [9]:
# your code here
# Generate mutual exclusion and exactly one conditions
mutual_excl_formula = mutually_exclusive(atoms)
exactly_one_formula = exactly_one_condition(atoms)

# Conjoin them into a larger formula and print
formulas = [mutual_excl_formula, exactly_one_formula]
conjoin_and_print(formulas)

(a <=> (-b & -c & -d)) & (b <=> (-a & -c & -d)) & (c <=> (-a & -b & -d)) & (d <=> (-a & -b & -c)) & 
(a v b v c v d) & ((-b & -c & -d) v (-a & -c & -d) v (-a & -b & -d) v (-a & -b & -c))



# 2. Bulding the Propositional atoms required for the puzzle

Let's dive into the formalization process. To express the constraints of the puzzle, we need to create propositional atoms. These atoms represent various properties associated with the houses and their inhabitants. For example, we use atoms like `blue1`, `brit3`, and `milk5` to indicate specific attributes such as the color of a house, the nationality of its resident, and the beverages they drink. We use `blue1` to denote the proposition "House 1 is Blue", `brit3` to denote "The resident of House 3 is British", `milk5` to say that "The resident of House5 drinks milk" and so on.

Let's start with the propositional atoms to express constraints on the possible colors of the houses. Each house can have one out of five colors, so we create $5\times 5$ propositional atoms to represent each possibility.


Try it:
Just execute the cells in this group, to make sure the relevant data structures with the atoms are generated. (If you aren't familiar with Pandas, don't worry. You will learn about Pandas very soon!)



In [10]:
number_of_houses = 5
colors = ["blue", "green", "red", "white", "yellow"]
house_colors = {}
for i in range(1, number_of_houses + 1):
    house_colors[i] = [f"{color}{i}" for color in colors]

# propositional atoms for the possible colors of house 1
house_colors[1]

['blue1', 'green1', 'red1', 'white1', 'yellow1']

In [11]:
import pandas as pd

color_atoms = pd.DataFrame(house_colors).T
color_atoms.index = ['HOUSE1', 'HOUSE2', 'HOUSE3', 'HOUSE4', 'HOUSE5']
color_atoms.columns = [color for color in colors]
color_atoms

,blue,green,red,white,yellow
HOUSE1,blue1,green1,red1,white1,yellow1
HOUSE2,blue2,green2,red2,white2,yellow2
HOUSE3,blue3,green3,red3,white3,yellow3
HOUSE4,blue4,green4,red4,white4,yellow4
HOUSE5,blue5,green5,red5,white5,yellow5


Now let's add atoms for the nationalities of the residents of the houses:

In [12]:
nationalities = ["brit", "dane", "german", "norwegian", "swede"]
house_nationalities = {}
for i in range(1, number_of_houses + 1):
    house_nationalities[i] = [f"{color}{i}" for color in nationalities]

nationalities_atoms = pd.DataFrame(house_nationalities).T
nationalities_atoms.index = ['HOUSE1', 'HOUSE2', 'HOUSE3', 'HOUSE4', 'HOUSE5']
nationalities_atoms.columns = nationalities
nationalities_atoms

,brit,dane,german,norwegian,swede
HOUSE1,brit1,dane1,german1,norwegian1,swede1
HOUSE2,brit2,dane2,german2,norwegian2,swede2
HOUSE3,brit3,dane3,german3,norwegian3,swede3
HOUSE4,brit4,dane4,german4,norwegian4,swede4
HOUSE5,brit5,dane5,german5,norwegian5,swede5


Next, for the beverages drank by the residents of the houses:

In [13]:
drinks = ["beer", "coffee", "milk", "tea", "water"]
house_drinks = {}
for i in range(1, number_of_houses + 1):
    house_drinks[i] = [f"{drink}{i}" for drink in drinks]

drinks_atoms = pd.DataFrame(house_drinks).T
drinks_atoms.index = ['HOUSE1', 'HOUSE2', 'HOUSE3', 'HOUSE4', 'HOUSE5']
drinks_atoms.columns = drinks
drinks_atoms

,beer,coffee,milk,tea,water
HOUSE1,beer1,coffee1,milk1,tea1,water1
HOUSE2,beer2,coffee2,milk2,tea2,water2
HOUSE3,beer3,coffee3,milk3,tea3,water3
HOUSE4,beer4,coffee4,milk4,tea4,water4
HOUSE5,beer5,coffee5,milk5,tea5,water5


Now, atoms for the cigarette brands smoked by the residents.

In [14]:
cigarette_brands = ["blends", "bluemaster", "dunhill", "pallmall", "prince"]
house_cigarette_brands = {}
for i in range(1, number_of_houses + 1):
    house_cigarette_brands[i] = [f"{brand}{i}" for brand in cigarette_brands]

cigarette_atoms = pd.DataFrame(house_cigarette_brands).T
cigarette_atoms.index = ['HOUSE1', 'HOUSE2', 'HOUSE3', 'HOUSE4', 'HOUSE5']
cigarette_atoms.columns = cigarette_brands
cigarette_atoms

,blends,bluemaster,dunhill,pallmall,prince
HOUSE1,blends1,bluemaster1,dunhill1,pallmall1,prince1
HOUSE2,blends2,bluemaster2,dunhill2,pallmall2,prince2
HOUSE3,blends3,bluemaster3,dunhill3,pallmall3,prince3
HOUSE4,blends4,bluemaster4,dunhill4,pallmall4,prince4
HOUSE5,blends5,bluemaster5,dunhill5,pallmall5,prince5


Finally, the atoms for the types of pets kept by the residents:

In [15]:
pet_types = ["birds", "cats", "dogs", "horses", "fish"]
house_pet_types = {}
for i in range(1, number_of_houses + 1):
    house_pet_types[i] = [f"{pet}{i}" for pet in pet_types]

pet_atoms = pd.DataFrame(house_pet_types).T
pet_atoms.index = ['HOUSE1', 'HOUSE2', 'HOUSE3', 'HOUSE4', 'HOUSE5']
pet_atoms.columns = pet_types
pet_atoms

,birds,cats,dogs,horses,fish
HOUSE1,birds1,cats1,dogs1,horses1,fish1
HOUSE2,birds2,cats2,dogs2,horses2,fish2
HOUSE3,birds3,cats3,dogs3,horses3,fish3
HOUSE4,birds4,cats4,dogs4,horses4,fish4
HOUSE5,birds5,cats5,dogs5,horses5,fish5


## Play a bit with your atoms

Before we get started with the formalization of the constraints, let's get a feeling for the atoms and the formulas we can express with them.

In [16]:
# Write a formula for "the brit does NOT live in house 1 OR (the fifth house is yellow AND the resident of the fourth house has dogs as pets)"
# You can write out this manually here, using ~ for negation, v for disjunction and & for conjunction
formula = f"(~brit1 v (yellow5 & dogs4))"
print(formula)

(~brit1 v (yellow5 & dogs4))


--------------------------------------
#3. Formalizing the constraints

We're now ready to write the logical statements that define the clues and constraints within the puzzle.

First, we need to encode the statement:

> Each of the five houses is painted a unique color (that is, no two houses have the same color). Each inhabitant of the house has a unique nationalities, type of pets, beverage they drink, and brand of cigarettes they smoke.

That is, we need to express that (1) each house has **one** color, nationality, drink, cigarette brand and pet type, and (2) each of these attributes is **unique** to that house.

For colors, we can capture this using two separate constraints:

**C1**: each house has only one color (house1 is not both blue and green, etc.).

**C2**: no two houses have the same color (house 1 and house 2 have different colors, etc.)

We'll define a separate formula for each such constraint.

In [17]:
mutual_excl_constraints = []
for house in range(1,number_of_houses+1):
  # each one of the five houses is painted in a single color
  mutual_excl_constraints.append(mutually_exclusive(house_colors[house]))
for house in range(1,number_of_houses+1):
  # each one of the five houses has a single nationality
  mutual_excl_constraints.append(mutually_exclusive(house_nationalities[house]))
for house in range(1,number_of_houses+1):
  # each one of the five houses drinks a single beverage
  mutual_excl_constraints.append(mutually_exclusive(house_drinks[house]))
for house in range(1,number_of_houses+1):
  # each one of the five houses smokes a single brand of cigarettes
  mutual_excl_constraints.append(mutually_exclusive(house_cigarette_brands[house]))
for house in range(1,number_of_houses+1):
  # each one of the five houses keeps a single type of pet
  mutual_excl_constraints.append(mutually_exclusive(house_pet_types[house]))

Try it: Run the cell above, and use the `conjoin_and_print` function to visualize the conjunction of the formulas we have stored in the list `mutual_excl_constraints`.

In [18]:
# your code here
conjoin_and_print(mutual_excl_constraints)

(blue1 <=> (-green1 & -red1 & -white1 & -yellow1)) & (green1 <=> (-blue1 & -red1 & -white1 & -yellow1)) & (red1 <=> (-blue1 & -green1 & -white1 & -yellow1)) & (white1 <=> (-blue1 & -green1 & -red1 & -yellow1)) & (yellow1 <=> (-blue1 & -green1 & -red1 & -white1)) & 
(blue2 <=> (-green2 & -red2 & -white2 & -yellow2)) & (green2 <=> (-blue2 & -red2 & -white2 & -yellow2)) & (red2 <=> (-blue2 & -green2 & -white2 & -yellow2)) & (white2 <=> (-blue2 & -green2 & -red2 & -yellow2)) & (yellow2 <=> (-blue2 & -green2 & -red2 & -white2)) & 
(blue3 <=> (-green3 & -red3 & -white3 & -yellow3)) & (green3 <=> (-blue3 & -red3 & -white3 & -yellow3)) & (red3 <=> (-blue3 & -green3 & -white3 & -yellow3)) & (white3 <=> (-blue3 & -green3 & -red3 & -yellow3)) & (yellow3 <=> (-blue3 & -green3 & -red3 & -white3)) & 
(blue4 <=> (-green4 & -red4 & -white4 & -yellow4)) & (green4 <=> (-blue4 & -red4 & -white4 & -yellow4)) & (red4 <=> (-blue4 & -green4 & -white4 & -yellow4)) & (white4 <=> (-blue4 & -green4 & -red4 & -ye

Mutual exclusivity constraints ensure that each house has one color, one pet, etc.

The following constraints ensure that no two houses have the same color, the same pet, etc.

In [19]:
exactly_one_constraints = []
for column,series in color_atoms.items():
  # each color occurs exactly once
  exactly_one_constraints.append(exactly_one_condition(series.tolist()))
for column,series in nationalities_atoms.items():
  # each nationality occurs exactly once
  exactly_one_constraints.append(exactly_one_condition(series.tolist()))
for column,series in drinks_atoms.items():
  # each drink occurs exactly once
  exactly_one_constraints.append(exactly_one_condition(series.tolist()))
for column,series in cigarette_atoms.items():
  # each cigarette brand occurs exactly once
  exactly_one_constraints.append(exactly_one_condition(series.tolist()))
for column,series in pet_atoms.items():
  # each pet type occurs exactly once
  exactly_one_constraints.append(exactly_one_condition(series.tolist()))

Try it: Run the cell above, and use the `conjoin_and_print` function to visualize the conjunction of the formulas we have stored in the list `exactly_one_constraints`.

In [20]:
# Your code here...
conjoin_and_print(exactly_one_constraints)

(blue1 v blue2 v blue3 v blue4 v blue5) & ((-blue2 & -blue3 & -blue4 & -blue5) v (-blue1 & -blue3 & -blue4 & -blue5) v (-blue1 & -blue2 & -blue4 & -blue5) v (-blue1 & -blue2 & -blue3 & -blue5) v (-blue1 & -blue2 & -blue3 & -blue4)) & 
(green1 v green2 v green3 v green4 v green5) & ((-green2 & -green3 & -green4 & -green5) v (-green1 & -green3 & -green4 & -green5) v (-green1 & -green2 & -green4 & -green5) v (-green1 & -green2 & -green3 & -green5) v (-green1 & -green2 & -green3 & -green4)) & 
(red1 v red2 v red3 v red4 v red5) & ((-red2 & -red3 & -red4 & -red5) v (-red1 & -red3 & -red4 & -red5) v (-red1 & -red2 & -red4 & -red5) v (-red1 & -red2 & -red3 & -red5) v (-red1 & -red2 & -red3 & -red4)) & 
(white1 v white2 v white3 v white4 v white5) & ((-white2 & -white3 & -white4 & -white5) v (-white1 & -white3 & -white4 & -white5) v (-white1 & -white2 & -white4 & -white5) v (-white1 & -white2 & -white3 & -white5) v (-white1 & -white2 & -white3 & -white4)) & 
(yellow1 v yellow2 v yellow3 v yell

# 4. Formalizing the clues
Now we're ready to formalize the 15 clues.

1. The Brit lives in the Red house.

2. The Swede keeps Dogs as pets.

3. The Dane drinks Tea.

4. The Green house is exactly to the left of the White house.

5. The owner of the Green house drinks Coffee.

6. The person who smokes Pall Mall rears Birds.

7. The owner of the Yellow house smokes Dunhill.

8. The man living in the centre house drinks Milk.

9. The Norwegian lives in the first house.

10. The man who smokes Blends lives next to the one who keeps Cats.

11. The man who keeps Horses lives next to the man who smokes Dunhill.

12. The man who smokes Blue Master drinks Beer.

13. The German smokes Prince.

14. The Norwegian lives next to the Blue house.

15. The man who smokes Blends has a neighbour who drinks Water.


## Other convenient functions the help on the different Clue types
Here, we can use some additional helper functions. Many cues have a similar structure, and these functions will allow us to exploit this and translate things into logic quickly.

Do not forget to run the cell below, after you have understood what they do....

In [21]:
def same_house_constraint(feature1,feature2):
  # house i has feature 1 if house i has feature 2
  return big_or([f"({feature1}{house} & {feature2}{house})" for house in range(1,number_of_houses+1)])

def house_left_constraint(feature1,feature2):
  # house i has feature1 if house i+1 has feature2
  return big_or([f"({feature1}{house} & {feature2}{house+1})" for house in range(1,number_of_houses)])

def house_adjacent_constraint(feature1,feature2):
  # house i has feature 1 if house i-1 or house i+1 has feature 2
  first_house = [f"({feature1}1 & {feature2}2)"]
  second_to_fourth_house = [f"({feature1}{house} & ({feature2}{house+1} v {feature2}{house-1}))" for house in range(2,number_of_houses)]
  fifth_house = [f"({feature1}5 & {feature2}4)"]
  return big_or(first_house+second_to_fourth_house+fifth_house)

# Create a "statements" list
statements = [0] * 15

## Time to Work! Lets code the  15 clues

I will do the first, you do the rest... ! Write them up and execute the cells to see the result.

In [22]:
# 1. The Brit lives in the Red house.
statements[0] = same_house_constraint("brit","red")
statements[0]

'((brit1 & red1) v (brit2 & red2) v (brit3 & red3) v (brit4 & red4) v (brit5 & red5))'

In [23]:
# 2. The Swede keeps Dogs as pets.
statements[1] = same_house_constraint("swede","dogs")
statements[1]

'((swede1 & dogs1) v (swede2 & dogs2) v (swede3 & dogs3) v (swede4 & dogs4) v (swede5 & dogs5))'

In [24]:
# 3. The Dane drinks Tea.
statements[2] = same_house_constraint("dane","tea")
statements[2]

'((dane1 & tea1) v (dane2 & tea2) v (dane3 & tea3) v (dane4 & tea4) v (dane5 & tea5))'

In [25]:
# 4. The Green house is exactly to the left of the White house.
statements[3] = house_left_constraint("green","white")
statements[3]

'((green1 & white2) v (green2 & white3) v (green3 & white4) v (green4 & white5))'

In [26]:
# 5. The owner of the Green house drinks Coffee.
statements[4] = same_house_constraint("green","coffee")
statements[4]

'((green1 & coffee1) v (green2 & coffee2) v (green3 & coffee3) v (green4 & coffee4) v (green5 & coffee5))'

In [27]:
# 6. The person who smokes Pall Mall rears Birds.
statements[5] = same_house_constraint("pallmall","birds")
statements[5]

'((pallmall1 & birds1) v (pallmall2 & birds2) v (pallmall3 & birds3) v (pallmall4 & birds4) v (pallmall5 & birds5))'

In [28]:
# 7. The owner of the Yellow house smokes Dunhill.
statements[6] = same_house_constraint("yellow","dunhill")
statements[6]

'((yellow1 & dunhill1) v (yellow2 & dunhill2) v (yellow3 & dunhill3) v (yellow4 & dunhill4) v (yellow5 & dunhill5))'

In [29]:
# 8. The man living in the centre house drinks Milk.
statements[7] = "milk3"  # Casa 3 es la central
statements[7]

'milk3'

In [30]:
# 9. The Norwegian lives in the first house.
statements[8] = "norwegian1"
statements[8]

'norwegian1'

In [31]:
# 10. The man who smokes Blends lives next to the one who keeps Cats.
statements[9] = house_adjacent_constraint("blends","cats")
statements[9]

'((blends1 & cats2) v (blends2 & (cats3 v cats1)) v (blends3 & (cats4 v cats2)) v (blends4 & (cats5 v cats3)) v (blends5 & cats4))'

In [32]:
# 11. The man who keeps Horses lives next to the man who smokes Dunhill.
statements[10] = house_adjacent_constraint("horses","dunhill")
statements[10]

'((horses1 & dunhill2) v (horses2 & (dunhill3 v dunhill1)) v (horses3 & (dunhill4 v dunhill2)) v (horses4 & (dunhill5 v dunhill3)) v (horses5 & dunhill4))'

In [33]:
# 12. The man who smokes Blue Master drinks Beer.
statements[11] = same_house_constraint("bluemaster","beer")
statements[11]

'((bluemaster1 & beer1) v (bluemaster2 & beer2) v (bluemaster3 & beer3) v (bluemaster4 & beer4) v (bluemaster5 & beer5))'

In [34]:
# 13. The German smokes Prince.
statements[12] = same_house_constraint("german","prince")
statements[12]

'((german1 & prince1) v (german2 & prince2) v (german3 & prince3) v (german4 & prince4) v (german5 & prince5))'

In [35]:
# 14. The Norwegian lives next to the Blue house.
statements[13] = house_adjacent_constraint("norwegian","blue")
statements[13]

'((norwegian1 & blue2) v (norwegian2 & (blue3 v blue1)) v (norwegian3 & (blue4 v blue2)) v (norwegian4 & (blue5 v blue3)) v (norwegian5 & blue4))'

In [36]:
# 15. The man who smokes Blends has a neighbour who drinks Water.
statements[14] = house_adjacent_constraint("blends","water")
statements[14]

'((blends1 & water2) v (blends2 & (water3 v water1)) v (blends3 & (water4 v water2)) v (blends4 & (water5 v water3)) v (blends5 & water4))'

In [37]:
# Print the complete "statements" record
statements

['((brit1 & red1) v (brit2 & red2) v (brit3 & red3) v (brit4 & red4) v (brit5 & red5))',
 '((swede1 & dogs1) v (swede2 & dogs2) v (swede3 & dogs3) v (swede4 & dogs4) v (swede5 & dogs5))',
 '((dane1 & tea1) v (dane2 & tea2) v (dane3 & tea3) v (dane4 & tea4) v (dane5 & tea5))',
 '((green1 & white2) v (green2 & white3) v (green3 & white4) v (green4 & white5))',
 '((green1 & coffee1) v (green2 & coffee2) v (green3 & coffee3) v (green4 & coffee4) v (green5 & coffee5))',
 '((pallmall1 & birds1) v (pallmall2 & birds2) v (pallmall3 & birds3) v (pallmall4 & birds4) v (pallmall5 & birds5))',
 '((yellow1 & dunhill1) v (yellow2 & dunhill2) v (yellow3 & dunhill3) v (yellow4 & dunhill4) v (yellow5 & dunhill5))',
 'milk3',
 'norwegian1',
 '((blends1 & cats2) v (blends2 & (cats3 v cats1)) v (blends3 & (cats4 v cats2)) v (blends4 & (cats5 v cats3)) v (blends5 & cats4))',
 '((horses1 & dunhill2) v (horses2 & (dunhill3 v dunhill1)) v (horses3 & (dunhill4 v dunhill2)) v (horses4 & (dunhill5 v dunhill3)) v

## Final assembly...
Finally, combine all your formulas into a single formula, using conjunction. For this, execute the next cell.


In [38]:
conjoin_and_print(mutual_excl_constraints+exactly_one_constraints+list(statements))

(blue1 <=> (-green1 & -red1 & -white1 & -yellow1)) & (green1 <=> (-blue1 & -red1 & -white1 & -yellow1)) & (red1 <=> (-blue1 & -green1 & -white1 & -yellow1)) & (white1 <=> (-blue1 & -green1 & -red1 & -yellow1)) & (yellow1 <=> (-blue1 & -green1 & -red1 & -white1)) & 
(blue2 <=> (-green2 & -red2 & -white2 & -yellow2)) & (green2 <=> (-blue2 & -red2 & -white2 & -yellow2)) & (red2 <=> (-blue2 & -green2 & -white2 & -yellow2)) & (white2 <=> (-blue2 & -green2 & -red2 & -yellow2)) & (yellow2 <=> (-blue2 & -green2 & -red2 & -white2)) & 
(blue3 <=> (-green3 & -red3 & -white3 & -yellow3)) & (green3 <=> (-blue3 & -red3 & -white3 & -yellow3)) & (red3 <=> (-blue3 & -green3 & -white3 & -yellow3)) & (white3 <=> (-blue3 & -green3 & -red3 & -yellow3)) & (yellow3 <=> (-blue3 & -green3 & -red3 & -white3)) & 
(blue4 <=> (-green4 & -red4 & -white4 & -yellow4)) & (green4 <=> (-blue4 & -red4 & -white4 & -yellow4)) & (red4 <=> (-blue4 & -green4 & -white4 & -yellow4)) & (white4 <=> (-blue4 & -green4 & -red4 & -ye

# 4. Solving the puzzle

We are ready to solve the puzzle! We can conjoin all our formulas into a large conjunction and pass it to an inference procedure, or solver.

Since our language involves 125 atoms, which leads to an incredibly large truth table with $2^{125}=42.535.295.865.117.307.932.921.825.928.971.026.432$ rows, we'll need something more efficient than the "naive"  approaches to model-checking and resolution that we saw in class.

State-of-the-art algorithms can solve problems involving tens of thousands of atoms and formulas consisting of millions of symbols. Our problem is not that large, though, and we can already get a solution in a fraction of a second using a classic algorithm called the Davis-Putnam-Logemann-Loveland (DPLL) algorithm. DPLL is a widely-used complete inference procedure; it underlies most modern solvers and uses a combination of search and resolution.

We'll use a DPLL solver provided through Logictools. You can access it via the following link:

[https://logictools.org/prop.html](https://logictools.org/prop.html)

Simply paste your formula there, click the "solve" button, and you'll receive an answer. If your formulas are correctly formulated, you'll get an answer in the following format:

````
Clause set is true if we assign values to variables as: ...
````

Copy the text that comes after the `:`, and paste it in the code block below, as the `solution_string`. This string contains the solution to the puzzle according to your formalization of the constraints.

Make sure to choose `dpll:better` as your solver.

You may be able to get a solution as well using an optimized version of resolution, labelled as `resolution:better`, or the optimized version of model-checking, called `truth tables:better`. If you try `resolution:naive` or `truth tables:naive`, which are the basic versions we saw in class, you are unlikely to get a solution.

In [39]:
# use dpll:better, copy the result you get and paste it here, as solution_string
solution_string = """
-blue1 -green1 -red1 -white1 yellow1 blue2 -green2 -red2 -white2 -yellow2 -blue3 -green3 red3 -white3 -yellow3 -blue4 green4 -red4 -white4 -yellow4 -blue5 -green5 -red5 white5 -yellow5 -brit1 -dane1 -german1 norwegian1 -swede1 -brit2 dane2 -german2 -norwegian2 -swede2 brit3 -dane3 -german3 -norwegian3 -swede3 -brit4 -dane4 german4 -norwegian4 -swede4 -brit5 -dane5 -german5 -norwegian5 swede5 -beer1 -coffee1 -milk1 -tea1 water1 -beer2 -coffee2 -milk2 tea2 -water2 -beer3 -coffee3 milk3 -tea3 -water3 -beer4 coffee4 -milk4 -tea4 -water4 beer5 -coffee5 -milk5 -tea5 -water5 -blends1 -bluemaster1 dunhill1 -pallmall1 -prince1 blends2 -bluemaster2 -dunhill2 -pallmall2 -prince2 -blends3 -bluemaster3 -dunhill3 pallmall3 -prince3 -blends4 -bluemaster4 -dunhill4 -pallmall4 prince4 -blends5 bluemaster5 -dunhill5 -pallmall5 -prince5 -birds1 cats1 -dogs1 -horses1 -fish1 -birds2 -cats2 -dogs2 horses2 -fish2 birds3 -cats3 -dogs3 -horses3 -fish3 -birds4 -cats4 -dogs4 -horses4 fish4 -birds5 -cats5 dogs5 -horses5 -fish5
"""
solution_string

'\n-blue1 -green1 -red1 -white1 yellow1 blue2 -green2 -red2 -white2 -yellow2 -blue3 -green3 red3 -white3 -yellow3 -blue4 green4 -red4 -white4 -yellow4 -blue5 -green5 -red5 white5 -yellow5 -brit1 -dane1 -german1 norwegian1 -swede1 -brit2 dane2 -german2 -norwegian2 -swede2 brit3 -dane3 -german3 -norwegian3 -swede3 -brit4 -dane4 german4 -norwegian4 -swede4 -brit5 -dane5 -german5 -norwegian5 swede5 -beer1 -coffee1 -milk1 -tea1 water1 -beer2 -coffee2 -milk2 tea2 -water2 -beer3 -coffee3 milk3 -tea3 -water3 -beer4 coffee4 -milk4 -tea4 -water4 beer5 -coffee5 -milk5 -tea5 -water5 -blends1 -bluemaster1 dunhill1 -pallmall1 -prince1 blends2 -bluemaster2 -dunhill2 -pallmall2 -prince2 -blends3 -bluemaster3 -dunhill3 pallmall3 -prince3 -blends4 -bluemaster4 -dunhill4 -pallmall4 prince4 -blends5 bluemaster5 -dunhill5 -pallmall5 -prince5 -birds1 cats1 -dogs1 -horses1 -fish1 -birds2 -cats2 -dogs2 horses2 -fish2 birds3 -cats3 -dogs3 -horses3 -fish3 -birds4 -cats4 -dogs4 -horses4 fish4 -birds5 -cats5 dogs

# 5. Lets visualize the solution

Finally, let's visualize your solution. We'll take that solution string and generate a table like the one you saw in the webpage containing the problem description:

https://www.brainzilla.com/logic/zebra/einsteins-riddle/

In [40]:
import pandas as pd

# Define the rows and columns
nationalities = ["brit", "dane", "german", "norwegian", "swede"]
colors = ["blue", "green", "red", "white", "yellow"]
drinks = ["beer", "coffee", "milk", "tea", "water"]
cigarette_brands = ["blends", "bluemaster", "dunhill", "pallmall", "prince"]
pet_types = ["birds", "cats", "dogs", "horses", "fish"]
rows = [colors, nationalities, drinks, cigarette_brands, pet_types]
rows_names = ["Color", "Nationality", "Drink", "Cigarette", "Pet"]
columns = ["House1", "House2", "House3", "House4", "House5"]

# Initialize an empty 5x5 DataFrame
df = pd.DataFrame(columns=columns, index=rows_names)

# Split the solution string into a list of literals
literals = solution_string.split()

# Parse the literals and populate the DataFrame
for literal in literals:
    # Check if the literal is true (no "-")
    if "-" not in literal:
        # Extract the category and house number from the literal
        category, value = literal[:-1], literal[-1]
        # Determine the row and column indices based on the category and value
        for i, row in enumerate(rows):
            if category in row:
                row_index = i
                column_index = int(value) - 1
                # Update the corresponding cell in the DataFrame
                df.iat[row_index, column_index] = category
df

,House1,House2,House3,House4,House5
Color,yellow,blue,red,green,white
Nationality,norwegian,dane,brit,german,swede
Drink,water,tea,milk,coffee,beer
Cigarette,dunhill,blends,pallmall,prince,bluemaster
Pet,cats,horses,birds,fish,dogs


To check if your solution is correct, revisit the webpage containing the problem description and fill up the provided table. It will provide feedback on whether your solution is all right. Good luck!